In [ ]:
import os
import json
from concurrent.futures import ProcessPoolExecutor

INPUT_DIR = "wiki_output"
OUTPUT_DIR = "wiki_paragraph_corpus"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def is_heading(text):
    """
    Detect Wikipedia section headings.
    Examples:
        History.
        Early life.
        See also.
        External links.
    """
    text = text.strip()

    if not text:
        return False

    words = text.split()

    if len(words) > 6:
        return False

    if not text.endswith("."):
        return False

    # ordinary sentences usually contain punctuation
    if any(x in text for x in [",", ";", ":", "?", "!"]):
        return False

    return True


def process_file(args):

    infile, outfile = args

    os.makedirs(os.path.dirname(outfile), exist_ok=True)

    with open(infile, "r", encoding="utf-8") as fin, \
         open(outfile, "w", encoding="utf-8") as fout:

        for line in fin:

            try:

                doc = json.loads(line)

                article_id = doc["id"]
                title = doc["title"].strip()
                contents = doc.get("contents", "").strip()

                ###################################################
                # Skip empty pages
                ###################################################

                if not contents:
                    continue

                ###################################################
                # Remove duplicated title if present
                ###################################################

                body = contents

                #if body.startswith(title):
                #    body = body[len(title):].strip()


                ###################################################
                # Split into paragraphs
                ###################################################

                raw_paragraphs = [
                    p.strip()
                    for p in body.split("\n")
                    if p.strip()
                ]

                ###################################################
                # Merge headings with next paragraph
                ###################################################

                paragraphs = []

                pending_headings = []

                for p in raw_paragraphs:

                    if is_heading(p):

                        pending_headings.append(p)
                        continue

                    if pending_headings:

                        merged = (
                            "\n\n".join(pending_headings)
                            + "\n\n"
                            + p
                        )

                        pending_headings = []

                    else:

                        merged = p

                    paragraphs.append(merged)

                ###################################################
                # Edge case:
                # article ends with heading(s)
                ###################################################

                if pending_headings:

                    paragraphs.append(
                        "\n\n".join(pending_headings)
                    )

                ###################################################
                # Write one JSON per paragraph
                ###################################################

                for paragraph_id, paragraph in enumerate(paragraphs):

                    out_doc = {

                        "passage_id": f"{article_id}_{paragraph_id}",
                        "article_id": article_id,
                        "paragraph_id": paragraph_id,

                        "title": title,

                        "url": doc.get("url", ""),

                        "contents": (
                            title
                            + "\n\n"
                            + paragraph
                        )
                    }

                    fout.write(
                        json.dumps(
                            out_doc,
                            ensure_ascii=False
                        )
                        + "\n"
                    )

            except Exception as e:

                print(f"ERROR in {infile}")
                print(e)


############################################################

jobs = []

for root, _, files in os.walk(INPUT_DIR):

    for file in files:

        if not file.endswith(".json"):
            continue

        infile = os.path.join(root, file)

        rel = os.path.relpath(infile, INPUT_DIR)

        outfile = os.path.join(OUTPUT_DIR, rel)

        jobs.append((infile, outfile))

print(f"Found {len(jobs)} files.")

############################################################

with ProcessPoolExecutor(max_workers=8) as executor:

    list(executor.map(process_file, jobs))

print("Finished creating paragraph corpus.")

In [ ]:
!pip install sentence-transformers

In [ ]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


# ============================================================
# Configuration
# ============================================================

INPUT_DIR = Path("wiki_paragraph_corpus")
OUTPUT_DIR = Path("wiki_embeddings")

MODEL_NAME = "intfloat/e5-base-v2"

# A100:
# 1024 is a safe starting point.
# You can try 1536 or 2048 later if GPU memory allows.
BATCH_SIZE = 1024

# Number of passages per saved embedding shard.
SHARD_SIZE = 100_000

# Float16 uses roughly half the storage of float32.
# Convert to float32 later when adding vectors to FAISS.
SAVE_DTYPE = np.float16

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Validate environment
# ============================================================

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Input directory does not exist: {INPUT_DIR.resolve()}"
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU was not detected.")

print(f"GPU: {torch.cuda.get_device_name(0)}")

# TF32 is useful on A100 for faster matrix multiplication.
torch.backends.cuda.matmul.allow_tf32 = True


# ============================================================
# Find input files in deterministic order
# ============================================================

json_files = sorted(
    path
    for path in INPUT_DIR.rglob("*.json")
    if path.is_file()
)

if not json_files:
    raise RuntimeError(
        f"No JSON files were found under {INPUT_DIR.resolve()}"
    )

print(f"Found {len(json_files):,} JSON files.")


# ============================================================
# Remove incomplete temporary shards
# ============================================================

for temp_dir in OUTPUT_DIR.glob("shard_*.tmp"):
    if temp_dir.is_dir():
        print(f"Removing incomplete temporary shard: {temp_dir}")
        shutil.rmtree(temp_dir)


# ============================================================
# Check whether a shard is complete
# ============================================================

def validate_completed_shard(shard_dir: Path) -> dict:
    """
    Validate a completed shard and return its manifest.
    """

    embedding_path = shard_dir / "embeddings.npy"
    metadata_path = shard_dir / "metadata.jsonl"
    manifest_path = shard_dir / "manifest.json"

    required_files = [
        embedding_path,
        metadata_path,
        manifest_path,
    ]

    missing = [
        str(path)
        for path in required_files
        if not path.exists()
    ]

    if missing:
        raise RuntimeError(
            f"Incomplete shard directory: {shard_dir}\n"
            f"Missing files: {missing}"
        )

    with open(manifest_path, "r", encoding="utf-8") as fin:
        manifest = json.load(fin)

    embeddings = np.load(
        embedding_path,
        mmap_mode="r",
    )

    expected_rows = int(manifest["rows"])
    expected_dimension = int(manifest["dimension"])

    if embeddings.shape != (expected_rows, expected_dimension):
        raise RuntimeError(
            f"Embedding shape does not match manifest in {shard_dir}.\n"
            f"Embeddings: {embeddings.shape}\n"
            f"Manifest: ({expected_rows}, {expected_dimension})"
        )

    return manifest


# ============================================================
# Resume from latest completed shard
# ============================================================

def load_resume_state() -> dict:
    """
    Find the latest completed shard and return the next source
    position from which embedding should continue.
    """

    completed_shards = sorted(
        path
        for path in OUTPUT_DIR.glob("shard_[0-9]*")
        if path.is_dir() and not path.name.endswith(".tmp")
    )

    if not completed_shards:
        return {
            "next_shard_id": 0,
            "next_file_index": 0,
            "next_line_index": 0,
            "total_passages": 0,
        }

    # Validate all existing shards so we do not resume from
    # corrupted output.
    manifests = []

    for shard_dir in completed_shards:
        manifest = validate_completed_shard(shard_dir)
        manifests.append((shard_dir, manifest))

    latest_dir, latest_manifest = manifests[-1]

    print(f"Latest completed shard: {latest_dir}")

    return {
        "next_shard_id": int(latest_manifest["shard_id"]) + 1,
        "next_file_index": int(
            latest_manifest["next_file_index"]
        ),
        "next_line_index": int(
            latest_manifest["next_line_index"]
        ),
        "total_passages": int(
            latest_manifest["total_passages"]
        ),
    }


resume_state = load_resume_state()

shard_id = resume_state["next_shard_id"]
start_file_index = resume_state["next_file_index"]
start_line_index = resume_state["next_line_index"]
total_passages = resume_state["total_passages"]

print(f"Next shard ID      : {shard_id:,}")
print(f"Next file index    : {start_file_index:,}")
print(f"Next line index    : {start_line_index:,}")
print(f"Already embedded   : {total_passages:,}")

if start_file_index > len(json_files):
    raise RuntimeError(
        "Resume file index is larger than the number of input files."
    )

if start_file_index == len(json_files):
    print("All input files have already been embedded.")
    raise SystemExit(0)


# ============================================================
# Load embedding model
# ============================================================

print(f"Loading model: {MODEL_NAME}")

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda",
)

model.max_seq_length = 512
model.half()
model.eval()

embedding_dimension = model.get_sentence_embedding_dimension()

print(f"Embedding dimension: {embedding_dimension}")
print(f"Batch size         : {BATCH_SIZE:,}")
print(f"Shard size         : {SHARD_SIZE:,}")


# ============================================================
# Save one embedding shard
# ============================================================

def save_shard(
    texts: list[str],
    metadata: list[dict],
    current_shard_id: int,
    next_file_index: int,
    next_line_index: int,
    cumulative_total: int,
) -> None:
    """
    Embed and atomically save one shard.

    A shard is first written to shard_XXXXX.tmp. It is renamed to
    shard_XXXXX only after every output file has been written.
    """

    if not texts:
        return

    if len(texts) != len(metadata):
        raise RuntimeError(
            "Text and metadata counts differ: "
            f"{len(texts):,} versus {len(metadata):,}"
        )

    final_dir = (
        OUTPUT_DIR / f"shard_{current_shard_id:05d}"
    )
    temp_dir = (
        OUTPUT_DIR / f"shard_{current_shard_id:05d}.tmp"
    )

    if final_dir.exists():
        raise RuntimeError(
            f"Final shard already exists: {final_dir}"
        )

    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    temp_dir.mkdir(parents=True)

    print()
    print(
        f"Embedding shard {current_shard_id:05d}: "
        f"{len(texts):,} passages"
    )

    with torch.inference_mode():
        embeddings = model.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

    if embeddings.ndim != 2:
        raise RuntimeError(
            f"Unexpected embedding dimensions: {embeddings.shape}"
        )

    if embeddings.shape[0] != len(texts):
        raise RuntimeError(
            "Embedding row count does not match passage count: "
            f"{embeddings.shape[0]:,} versus {len(texts):,}"
        )

    if embeddings.shape[1] != embedding_dimension:
        raise RuntimeError(
            "Unexpected embedding dimension: "
            f"{embeddings.shape[1]} instead of "
            f"{embedding_dimension}"
        )

    embeddings = embeddings.astype(
        SAVE_DTYPE,
        copy=False,
    )

    temporary_embedding_path = (
        temp_dir / "embeddings.npy"
    )
    temporary_metadata_path = (
        temp_dir / "metadata.jsonl"
    )
    temporary_manifest_path = (
        temp_dir / "manifest.json"
    )

    # Save embeddings.
    np.save(
        temporary_embedding_path,
        embeddings,
    )

    # Save metadata in exactly the same row order.
    with open(
        temporary_metadata_path,
        "w",
        encoding="utf-8",
    ) as fout:
        for item in metadata:
            fout.write(
                json.dumps(
                    item,
                    ensure_ascii=False,
                )
                + "\n"
            )

    manifest = {
        "shard_id": current_shard_id,
        "rows": int(embeddings.shape[0]),
        "dimension": int(embeddings.shape[1]),
        "dtype": str(embeddings.dtype),
        "model": MODEL_NAME,
        "normalized": True,
        "batch_size": BATCH_SIZE,
        "next_file_index": next_file_index,
        "next_line_index": next_line_index,
        "total_passages": cumulative_total,
    }

    with open(
        temporary_manifest_path,
        "w",
        encoding="utf-8",
    ) as fout:
        json.dump(
            manifest,
            fout,
            indent=2,
        )

    # Confirm the temporary files exist before renaming.
    if not temporary_embedding_path.exists():
        raise RuntimeError(
            f"Embedding file was not written: "
            f"{temporary_embedding_path}"
        )

    if not temporary_metadata_path.exists():
        raise RuntimeError(
            f"Metadata file was not written: "
            f"{temporary_metadata_path}"
        )

    if not temporary_manifest_path.exists():
        raise RuntimeError(
            f"Manifest file was not written: "
            f"{temporary_manifest_path}"
        )

    # Atomically mark the shard as complete.
    os.replace(
        temp_dir,
        final_dir,
    )

    # IMPORTANT:
    # After the rename, the old temporary path no longer exists.
    final_embedding_path = (
        final_dir / "embeddings.npy"
    )

    size_gb = (
        final_embedding_path.stat().st_size
        / (1024 ** 3)
    )

    print(
        f"Saved {final_dir} | "
        f"shape={embeddings.shape} | "
        f"dtype={embeddings.dtype} | "
        f"embedding size={size_gb:.3f} GB"
    )


# ============================================================
# Stream through corpus
# ============================================================

texts: list[str] = []
metadata: list[dict] = []

next_file_index = start_file_index
next_line_index = start_line_index

bad_json_lines = 0
empty_passages = 0


for file_index in range(
    start_file_index,
    len(json_files),
):
    infile = json_files[file_index]

    relative_source = str(
        infile.relative_to(INPUT_DIR)
    )

    # Only the first resumed file may start at a nonzero line.
    if file_index == start_file_index:
        line_to_start = start_line_index
    else:
        line_to_start = 0

    with open(
        infile,
        "r",
        encoding="utf-8",
    ) as fin:
        for line_index, line in enumerate(fin):
            if line_index < line_to_start:
                continue

            stripped_line = line.strip()

            if not stripped_line:
                next_file_index = file_index
                next_line_index = line_index + 1
                continue

            try:
                doc = json.loads(stripped_line)
            except json.JSONDecodeError as error:
                bad_json_lines += 1

                print(
                    f"WARNING: invalid JSON skipped: "
                    f"{infile}, line {line_index + 1}: "
                    f"{error}"
                )

                next_file_index = file_index
                next_line_index = line_index + 1
                continue

            contents = str(
                doc.get("contents", "")
            ).strip()

            if not contents:
                empty_passages += 1

                next_file_index = file_index
                next_line_index = line_index + 1
                continue

            # E5 corpus passages should use the "passage:" prefix.
            texts.append(
                "passage: " + contents
            )

            # Metadata row N corresponds to embedding row N.
            metadata.append({
                "passage_id": doc.get("passage_id"),
                "article_id": doc.get("article_id"),
                "paragraph_id": doc.get("paragraph_id"),
                "title": doc.get("title", ""),
                "url": doc.get("url", ""),
                "contents": contents,
                "source_file": relative_source,
                "source_line": line_index,
            })

            total_passages += 1

            # This cursor points to the next unread source line.
            next_file_index = file_index
            next_line_index = line_index + 1

            if len(texts) >= SHARD_SIZE:
                save_shard(
                    texts=texts,
                    metadata=metadata,
                    current_shard_id=shard_id,
                    next_file_index=next_file_index,
                    next_line_index=next_line_index,
                    cumulative_total=total_passages,
                )

                texts = []
                metadata = []
                shard_id += 1

    # The entire current file was consumed.
    next_file_index = file_index + 1
    next_line_index = 0

    if (file_index + 1) % 100 == 0:
        print(
            f"Files processed: {file_index + 1:,}/"
            f"{len(json_files):,} | "
            f"passages: {total_passages:,} | "
            f"next shard: {shard_id:05d}"
        )


# ============================================================
# Save final partial shard
# ============================================================

if texts:
    save_shard(
        texts=texts,
        metadata=metadata,
        current_shard_id=shard_id,
        next_file_index=len(json_files),
        next_line_index=0,
        cumulative_total=total_passages,
    )


# ============================================================
# Final report
# ============================================================

print()
print("Embedding completed.")
print(f"Total passages embedded : {total_passages:,}")
print(f"Invalid JSON lines       : {bad_json_lines:,}")
print(f"Empty passages skipped   : {empty_passages:,}")
print(f"Output directory         : {OUTPUT_DIR.resolve()}")

In [ ]:
!python -m pip install faiss-gpu

In [ ]:
import faiss

print("FAISS version:", faiss.__version__)
print("GPU support:", hasattr(faiss, "StandardGpuResources"))
print("GPU count:", faiss.get_num_gpus() if hasattr(faiss, "get_num_gpus") else 0)

In [ ]:
!python -m pip install tqdm

In [ ]:
import gc
import json
import math
import os
import shutil
import time
from pathlib import Path

import faiss
import numpy as np
from tqdm.auto import tqdm


# ============================================================
# Configuration
# ============================================================

EMBEDDING_DIR = Path("wiki_embeddings")
OUTPUT_DIR = Path("wiki_faiss_gpu")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_INDEX_PATH = OUTPUT_DIR / "wiki_e5_ivfpq.faiss"
OFFSETS_PATH = OUTPUT_DIR / "shard_offsets.json"
BUILD_INFO_PATH = OUTPUT_DIR / "build_info.json"

DIMENSION = 768

# IVF coarse clusters.
NLIST = 16_384

# 768 / 64 = 12 dimensions per subquantizer.
# With NBITS=8, each vector gets a 64-byte PQ code.
M = 64
NBITS = 8

# 40 training vectors per IVF centroid.
# 16,384 * 40 = 655,360.
TRAIN_SAMPLE_SIZE = 655_360

# Batch size when transferring vectors to FAISS GPU.
ADD_BATCH_SIZE = 65_536

# Default search setting.
NPROBE = 64

GPU_ID = 0
RANDOM_SEED = 42

# Set to 0 to disable checkpoints.
# Checkpoints take time because the entire GPU index must be copied to CPU.
CHECKPOINT_EVERY_SHARDS = 100

# Remove checkpoint after successful completion.
DELETE_CHECKPOINT_AFTER_SUCCESS = False

CHECKPOINT_INDEX_PATH = OUTPUT_DIR / "checkpoint.faiss"
CHECKPOINT_STATE_PATH = OUTPUT_DIR / "checkpoint_state.json"


# ============================================================
# Utility functions
# ============================================================

def format_duration(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours:
        return f"{hours:d}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes:d}m {seconds:02d}s"
    return f"{seconds:d}s"


def atomic_write_json(data: dict, destination: Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".tmp")

    with open(temporary, "w", encoding="utf-8") as fout:
        json.dump(data, fout, ensure_ascii=False, indent=2)

    os.replace(temporary, destination)


def save_faiss_index_atomic(index, destination: Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".tmp")

    if temporary.exists():
        temporary.unlink()

    faiss.write_index(index, str(temporary))
    os.replace(temporary, destination)


# ============================================================
# Validate FAISS GPU installation
# ============================================================

print(f"FAISS version: {faiss.__version__}")

if not hasattr(faiss, "StandardGpuResources"):
    raise RuntimeError(
        "This FAISS installation has no GPU support.\n"
        "Install a GPU-enabled FAISS build."
    )

gpu_count = faiss.get_num_gpus()
print(f"Available GPUs: {gpu_count}")

if gpu_count <= GPU_ID:
    raise RuntimeError(
        f"Requested GPU {GPU_ID}, but FAISS detected "
        f"{gpu_count} GPU(s)."
    )

if DIMENSION % M != 0:
    raise ValueError(
        f"DIMENSION={DIMENSION} must be divisible by M={M}."
    )

if NPROBE > 2048:
    raise ValueError("GPU FAISS requires NPROBE <= 2048.")


# ============================================================
# Discover and validate embedding shards
# ============================================================

shard_dirs = sorted(
    path
    for path in EMBEDDING_DIR.glob("shard_[0-9]*")
    if path.is_dir()
)

if not shard_dirs:
    raise RuntimeError(
        f"No embedding shards found in {EMBEDDING_DIR.resolve()}"
    )

shards = []
total_vectors = 0

print()
print("Validating embedding shards...")

for shard_dir in tqdm(
    shard_dirs,
    desc="Validating shards",
    unit="shard",
):
    embedding_path = shard_dir / "embeddings.npy"
    metadata_path = shard_dir / "metadata.jsonl"
    manifest_path = shard_dir / "manifest.json"

    required_paths = (
        embedding_path,
        metadata_path,
        manifest_path,
    )

    for required_path in required_paths:
        if not required_path.exists():
            raise RuntimeError(
                f"Incomplete shard {shard_dir}: "
                f"missing {required_path.name}"
            )

    with open(manifest_path, "r", encoding="utf-8") as fin:
        manifest = json.load(fin)

    rows = int(manifest["rows"])
    dimension = int(manifest["dimension"])

    if dimension != DIMENSION:
        raise RuntimeError(
            f"{shard_dir.name} has dimension {dimension}; "
            f"expected {DIMENSION}."
        )

    vectors = np.load(embedding_path, mmap_mode="r")

    if vectors.shape != (rows, DIMENSION):
        raise RuntimeError(
            f"Shape mismatch in {embedding_path}: "
            f"{vectors.shape}, expected {(rows, DIMENSION)}."
        )

    if vectors.dtype not in (np.float16, np.float32):
        raise RuntimeError(
            f"Unsupported dtype in {embedding_path}: {vectors.dtype}"
        )

    shards.append({
        "name": shard_dir.name,
        "embedding_path": embedding_path,
        "metadata_path": metadata_path,
        "rows": rows,
        "global_start": total_vectors,
        "global_end": total_vectors + rows,
    })

    total_vectors += rows

    del vectors

print()
print(f"Completed embedding shards: {len(shards):,}")
print(f"Total vectors             : {total_vectors:,}")
print(f"Embedding dimension       : {DIMENSION}")
print(f"IVF centroids             : {NLIST:,}")
print(f"PQ subquantizers          : {M}")
print(f"PQ code size              : {M * NBITS // 8} bytes/vector")
print(f"Training sample target    : {TRAIN_SAMPLE_SIZE:,}")


# ============================================================
# Create fast distributed training sample
# ============================================================

def create_training_sample() -> np.ndarray:
    """
    Take one randomly located contiguous block from every shard.

    This avoids slow random fancy indexing into memory-mapped arrays.
    The final combined sample is shuffled in RAM.
    """

    sample_size = min(TRAIN_SAMPLE_SIZE, total_vectors)

    if sample_size < NLIST:
        raise RuntimeError(
            f"Training sample {sample_size:,} is smaller than "
            f"NLIST={NLIST:,}."
        )

    rng = np.random.default_rng(RANDOM_SEED)

    # Allocate proportional sample counts.
    exact_counts = np.array(
        [
            sample_size * shard["rows"] / total_vectors
            for shard in shards
        ],
        dtype=np.float64,
    )

    sample_counts = np.floor(exact_counts).astype(np.int64)

    remaining = sample_size - int(sample_counts.sum())

    if remaining > 0:
        fractional_order = np.argsort(
            -(exact_counts - sample_counts)
        )
        sample_counts[fractional_order[:remaining]] += 1

    for index, shard in enumerate(shards):
        sample_counts[index] = min(
            int(sample_counts[index]),
            int(shard["rows"]),
        )

    actual_sample_size = int(sample_counts.sum())

    training_vectors = np.empty(
        (actual_sample_size, DIMENSION),
        dtype=np.float32,
    )

    write_position = 0
    sampling_started = time.time()

    print()
    print(
        f"Sampling {actual_sample_size:,} training vectors "
        f"using contiguous shard reads..."
    )

    progress = tqdm(
        zip(shards, sample_counts),
        total=len(shards),
        desc="Sampling training vectors",
        unit="shard",
    )

    for shard, take_value in progress:
        take = int(take_value)

        if take <= 0:
            continue

        rows = int(shard["rows"])

        vectors = np.load(
            shard["embedding_path"],
            mmap_mode="r",
        )

        maximum_start = rows - take

        if maximum_start > 0:
            start = int(
                rng.integers(
                    low=0,
                    high=maximum_start + 1,
                )
            )
        else:
            start = 0

        end = start + take

        training_vectors[
            write_position:write_position + take
        ] = np.asarray(
            vectors[start:end],
            dtype=np.float32,
        )

        write_position += take

        progress.set_postfix(
            sampled=f"{write_position:,}"
        )

        del vectors

    if write_position != actual_sample_size:
        training_vectors = training_vectors[:write_position]

    print("Shuffling training matrix...")
    rng.shuffle(training_vectors)

    elapsed = time.time() - sampling_started

    print(
        f"Training matrix: {training_vectors.shape}, "
        f"{training_vectors.nbytes / (1024 ** 3):.2f} GiB"
    )
    print(f"Sampling time  : {format_duration(elapsed)}")

    return np.ascontiguousarray(
        training_vectors,
        dtype=np.float32,
    )


# ============================================================
# Configure GPU resources
# ============================================================

gpu_resources = faiss.StandardGpuResources()

# FAISS normally reserves up to roughly 1.5 GiB for temporary work.
# This can be lowered if necessary:
# gpu_resources.setTempMemory(1024 * 1024 * 1024)

clone_options = faiss.GpuClonerOptions()

# Required for GPU IVFPQ codes of 56 bytes or more.
clone_options.useFloat16 = True

# Explicitly retain 64-bit IDs.
clone_options.indicesOptions = faiss.INDICES_64_BIT


# ============================================================
# Resume from checkpoint or train a new index
# ============================================================

start_shard_index = 0
next_global_id = 0
offset_records = []

checkpoint_available = (
    CHECKPOINT_INDEX_PATH.exists()
    and CHECKPOINT_STATE_PATH.exists()
)

if checkpoint_available:
    print()
    print("Checkpoint found. Loading checkpoint...")

    with open(
        CHECKPOINT_STATE_PATH,
        "r",
        encoding="utf-8",
    ) as fin:
        checkpoint_state = json.load(fin)

    expected_settings = {
        "dimension": DIMENSION,
        "nlist": NLIST,
        "m": M,
        "nbits": NBITS,
        "total_vectors": total_vectors,
        "shard_count": len(shards),
    }

    for key, expected_value in expected_settings.items():
        actual_value = checkpoint_state.get(key)

        if actual_value != expected_value:
            raise RuntimeError(
                f"Checkpoint setting mismatch for {key}: "
                f"{actual_value} != {expected_value}"
            )

    start_shard_index = int(
        checkpoint_state["next_shard_index"]
    )
    next_global_id = int(
        checkpoint_state["next_global_id"]
    )
    offset_records = list(
        checkpoint_state["offset_records"]
    )

    checkpoint_cpu_index = faiss.read_index(
        str(CHECKPOINT_INDEX_PATH)
    )

    if checkpoint_cpu_index.ntotal != next_global_id:
        raise RuntimeError(
            "Checkpoint index size does not match checkpoint state: "
            f"{checkpoint_cpu_index.ntotal:,} versus "
            f"{next_global_id:,}."
        )

    print(
        f"Resuming after shard {start_shard_index:,}; "
        f"{next_global_id:,} vectors already indexed."
    )

    print("Copying checkpoint index to GPU...")

    gpu_index = faiss.index_cpu_to_gpu(
        gpu_resources,
        GPU_ID,
        checkpoint_cpu_index,
        clone_options,
    )

    del checkpoint_cpu_index
    gc.collect()

else:
    training_vectors = create_training_sample()

    print()
    print("Creating CPU IVF-PQ structure...")

    cpu_quantizer = faiss.IndexFlatIP(DIMENSION)

    cpu_index = faiss.IndexIVFPQ(
        cpu_quantizer,
        DIMENSION,
        NLIST,
        M,
        NBITS,
        faiss.METRIC_INNER_PRODUCT,
    )

    cpu_index.nprobe = NPROBE

    print("Copying untrained index structure to GPU...")

    gpu_index = faiss.index_cpu_to_gpu(
        gpu_resources,
        GPU_ID,
        cpu_index,
        clone_options,
    )

    gpu_index.nprobe = NPROBE

    print()
    print("Training IVF-PQ index on GPU...")
    print(
        "FAISS does not expose a training progress callback; "
        "the next message appears when training completes."
    )

    training_started = time.time()

    gpu_index.train(training_vectors)

    training_elapsed = time.time() - training_started

    if not gpu_index.is_trained:
        raise RuntimeError("GPU IVF-PQ training failed.")

    print(
        f"GPU training completed in "
        f"{format_duration(training_elapsed)}."
    )

    del training_vectors
    del cpu_index
    del cpu_quantizer

    gc.collect()


# ============================================================
# Add all vectors
# ============================================================

gpu_index.nprobe = NPROBE

if not gpu_index.is_trained:
    raise RuntimeError("Cannot add vectors to an untrained index.")

if gpu_index.ntotal != next_global_id:
    raise RuntimeError(
        f"GPU index contains {gpu_index.ntotal:,} vectors, "
        f"but resume state expects {next_global_id:,}."
    )

build_started = time.time()

remaining_vectors = total_vectors - next_global_id

print()
print(
    f"Adding {remaining_vectors:,} remaining vectors "
    f"from shard {start_shard_index + 1:,}/"
    f"{len(shards):,}..."
)

overall_progress = tqdm(
    total=remaining_vectors,
    desc="Adding vectors",
    unit="vec",
    unit_scale=True,
    initial=0,
)

for shard_index in range(start_shard_index, len(shards)):
    shard = shards[shard_index]

    vectors = np.load(
        shard["embedding_path"],
        mmap_mode="r",
    )

    rows = int(shard["rows"])
    expected_global_start = int(shard["global_start"])

    if next_global_id != expected_global_start:
        raise RuntimeError(
            f"ID alignment error before {shard['name']}: "
            f"next ID is {next_global_id:,}, expected "
            f"{expected_global_start:,}."
        )

    shard_global_start = next_global_id

    batch_progress = tqdm(
        range(0, rows, ADD_BATCH_SIZE),
        desc=shard["name"],
        unit="batch",
        leave=False,
    )

    for batch_start in batch_progress:
        batch_end = min(
            batch_start + ADD_BATCH_SIZE,
            rows,
        )

        batch_vectors = np.ascontiguousarray(
            vectors[batch_start:batch_end],
            dtype=np.float32,
        )

        batch_count = int(batch_vectors.shape[0])

        batch_ids = np.arange(
            next_global_id,
            next_global_id + batch_count,
            dtype=np.int64,
        )

        gpu_index.add_with_ids(
            batch_vectors,
            batch_ids,
        )

        next_global_id += batch_count
        overall_progress.update(batch_count)

        batch_progress.set_postfix(
            total=f"{next_global_id:,}"
        )

        del batch_vectors
        del batch_ids

    del vectors

    offset_records.append({
        "shard_name": shard["name"],
        "metadata_path": str(shard["metadata_path"]),
        "rows": rows,
        "global_start": shard_global_start,
        "global_end": next_global_id,
    })

    completed_shards = shard_index + 1

    should_checkpoint = (
        CHECKPOINT_EVERY_SHARDS > 0
        and completed_shards < len(shards)
        and completed_shards % CHECKPOINT_EVERY_SHARDS == 0
    )

    if should_checkpoint:
        overall_progress.write(
            f"Creating checkpoint after "
            f"{completed_shards:,} shards..."
        )

        checkpoint_started = time.time()

        checkpoint_cpu_index = faiss.index_gpu_to_cpu(
            gpu_index
        )
        checkpoint_cpu_index.nprobe = NPROBE

        save_faiss_index_atomic(
            checkpoint_cpu_index,
            CHECKPOINT_INDEX_PATH,
        )

        checkpoint_state = {
            "dimension": DIMENSION,
            "nlist": NLIST,
            "m": M,
            "nbits": NBITS,
            "total_vectors": total_vectors,
            "shard_count": len(shards),
            "next_shard_index": completed_shards,
            "next_global_id": next_global_id,
            "offset_records": offset_records,
        }

        atomic_write_json(
            checkpoint_state,
            CHECKPOINT_STATE_PATH,
        )

        del checkpoint_cpu_index
        gc.collect()

        checkpoint_elapsed = (
            time.time() - checkpoint_started
        )

        overall_progress.write(
            f"Checkpoint completed in "
            f"{format_duration(checkpoint_elapsed)}."
        )

overall_progress.close()


# ============================================================
# Validate completed GPU index
# ============================================================

if next_global_id != total_vectors:
    raise RuntimeError(
        f"Added {next_global_id:,} vectors; "
        f"expected {total_vectors:,}."
    )

if gpu_index.ntotal != total_vectors:
    raise RuntimeError(
        f"GPU index contains {gpu_index.ntotal:,} vectors; "
        f"expected {total_vectors:,}."
    )

print()
print(f"All {total_vectors:,} vectors added successfully.")


# ============================================================
# Convert to CPU and save final index
# ============================================================

print("Converting final GPU index to CPU...")

conversion_started = time.time()

final_cpu_index = faiss.index_gpu_to_cpu(gpu_index)
final_cpu_index.nprobe = NPROBE

conversion_elapsed = time.time() - conversion_started

print(
    f"GPU-to-CPU conversion completed in "
    f"{format_duration(conversion_elapsed)}."
)

if final_cpu_index.ntotal != total_vectors:
    raise RuntimeError(
        f"Final CPU index contains "
        f"{final_cpu_index.ntotal:,} vectors; "
        f"expected {total_vectors:,}."
    )

print(f"Writing final index to {FINAL_INDEX_PATH}...")

save_started = time.time()

save_faiss_index_atomic(
    final_cpu_index,
    FINAL_INDEX_PATH,
)

save_elapsed = time.time() - save_started

print(
    f"Final index written in "
    f"{format_duration(save_elapsed)}."
)


# ============================================================
# Save metadata mapping and build information
# ============================================================

mapping = {
    "total_vectors": total_vectors,
    "dimension": DIMENSION,
    "model": "intfloat/e5-base-v2",
    "normalized_embeddings": True,
    "metric": "inner_product",
    "index_type": "IndexIVFPQ",
    "nlist": NLIST,
    "m": M,
    "nbits": NBITS,
    "default_nprobe": NPROBE,
    "shards": offset_records,
}

atomic_write_json(mapping, OFFSETS_PATH)

build_info = {
    "faiss_version": faiss.__version__,
    "gpu_id": GPU_ID,
    "total_vectors": total_vectors,
    "shard_count": len(shards),
    "dimension": DIMENSION,
    "nlist": NLIST,
    "m": M,
    "nbits": NBITS,
    "training_sample_size": min(
        TRAIN_SAMPLE_SIZE,
        total_vectors,
    ),
    "add_batch_size": ADD_BATCH_SIZE,
    "nprobe": NPROBE,
    "final_index_path": str(FINAL_INDEX_PATH.resolve()),
    "offsets_path": str(OFFSETS_PATH.resolve()),
}

atomic_write_json(build_info, BUILD_INFO_PATH)


# ============================================================
# Optional final sanity search
# ============================================================

print("Running final sanity search...")

first_vectors = np.load(
    shards[0]["embedding_path"],
    mmap_mode="r",
)

test_queries = np.ascontiguousarray(
    first_vectors[:5],
    dtype=np.float32,
)

distances, identifiers = final_cpu_index.search(
    test_queries,
    5,
)

if identifiers.shape != (5, 5):
    raise RuntimeError(
        f"Unexpected sanity-search shape: {identifiers.shape}"
    )

if np.any(identifiers[:, 0] < 0):
    raise RuntimeError(
        "Sanity search returned invalid nearest-neighbor IDs."
    )

print("Sanity search passed.")
print("Top IDs:")
print(identifiers)


# ============================================================
# Final report
# ============================================================

index_size_gib = (
    FINAL_INDEX_PATH.stat().st_size
    / (1024 ** 3)
)

total_build_elapsed = time.time() - build_started

print()
print("=" * 60)
print("GPU FAISS build completed")
print("=" * 60)
print(f"Vectors indexed : {final_cpu_index.ntotal:,}")
print(f"Index file      : {FINAL_INDEX_PATH.resolve()}")
print(f"Index size      : {index_size_gib:.2f} GiB")
print(f"Offsets file    : {OFFSETS_PATH.resolve()}")
print(f"Build info      : {BUILD_INFO_PATH.resolve()}")
print(
    f"Add/finalize time: "
    f"{format_duration(total_build_elapsed)}"
)

if (
    DELETE_CHECKPOINT_AFTER_SUCCESS
    and CHECKPOINT_INDEX_PATH.exists()
):
    CHECKPOINT_INDEX_PATH.unlink()

if (
    DELETE_CHECKPOINT_AFTER_SUCCESS
    and CHECKPOINT_STATE_PATH.exists()
):
    CHECKPOINT_STATE_PATH.unlink()

In [ ]:
from pathlib import Path
import faiss
import json

index_path = Path("wiki_faiss_gpu/wiki_e5_ivfpq.faiss")
offsets_path = Path("wiki_faiss_gpu/shard_offsets.json")

print("Index exists:", index_path.exists())
print("Offsets exist:", offsets_path.exists())

index = faiss.read_index(str(index_path))

print("Index vectors:", index.ntotal)
print("Index trained:", index.is_trained)

with open(offsets_path, "r", encoding="utf-8") as f:
    offsets = json.load(f)

print("Mapped vectors:", offsets["total_vectors"])